In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### Input & Output State

In [2]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

model = ChatOpenAI()

In [3]:
from typing import TypedDict


class ChatMessages(TypedDict):
    question: str
    answer: str
    llm_calls: int

In [4]:
def call_model(state: ChatMessages):
    question = state["question"]
    llm_calls = state.get("llm_calls", 0)
    state["llm_calls"] = llm_calls + 1
    print("LLM_CALLS:", state["llm_calls"])
    response = model.invoke(input=question)
    state["answer"] = response.content
    return state

In [5]:
workflow = StateGraph(ChatMessages)

workflow.add_edge(START, "agent")
workflow.add_node("agent", call_model)
workflow.add_edge("agent", END)

graph = workflow.compile()

In [6]:
graph.invoke(input={"question": "Whats the highest mountain in the world?"})

LLM_CALLS: 1


{'question': 'Whats the highest mountain in the world?',
 'answer': 'Mount Everest',
 'llm_calls': 1}

In [8]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class InputState(TypedDict):
    question: str


class PrivateState(TypedDict):
    llm_calls: int


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, PrivateState, OutputState):
    pass

In [9]:
workflow = StateGraph(state_schema=OverallState, input_schema=InputState, output_schema=OutputState)

workflow.add_edge(START, "agent")
workflow.add_node("agent", call_model)
workflow.add_edge("agent", END)

graph = workflow.compile()

In [10]:
graph.invoke({"question": "Whats the highest mountain in the world?"})

LLM_CALLS: 1


{'answer': 'Mount Everest, located in the Himalayas, is the highest mountain in the world, with a peak reaching an elevation of 29,032 feet (8,848 meters) above sea level.'}

### Add runtime configuration

In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables.config import RunnableConfig
from langchain_core.messages import SystemMessage, HumanMessage


def call_model(state: OverallState, config: RunnableConfig):
    language = config["configurable"].get("language", "English")
    system_message_content = "Respond in {}".format(language)
    system_message = SystemMessage(content=system_message_content)
    messages = [system_message, HumanMessage(content=state["question"])]
    response = model.invoke(messages)
    return {"answer": response}

In [13]:
workflow = StateGraph(ChatMessages)

workflow.add_edge(START, "agent")
workflow.add_node("agent", call_model)
workflow.add_edge("agent", END)

graph = workflow.compile()

In [14]:
config = {"configurable": {"language": "Spanish"}}
graph.invoke({"question": "What's the highest mountain in the world?"}, config=config)

{'question': "What's the highest mountain in the world?",
 'answer': AIMessage(content='La montaña más alta del mundo es el Monte Everest, que se encuentra en la cordillera del Himalaya, en la frontera entre Nepal y China. Alcanza una altitud de 8,848 metros sobre el nivel del mar.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 23, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dz8CCy1OPZSAVG51Q3HbirbY9GrjV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--8ebd1457-7856-4be6-9b86-e0039a34d5b4-0', usage_metadata={'input_tokens': 23, 'output_tokens': 51, 'total_tokens': 74, 'input_token_details': {'audio

In [15]:
config = {"configurable": {"language": "German"}}
graph.invoke({"question": "What's the highest mountain in the world?"}, config=config)

{'question': "What's the highest mountain in the world?",
 'answer': AIMessage(content='Der höchste Berg der Welt ist der Mount Everest in der Himalaya-Region.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 23, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dz8CQDQyU3j65xC0qHy61nQ0W2W8C', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--e7e1ef8c-2686-433a-bf5f-892b0902ac99-0', usage_metadata={'input_tokens': 23, 'output_tokens': 18, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})}

### Runtime Configuration >= LangGraph 0.6.0

In [16]:
from dataclasses import dataclass

@dataclass
class Context:
    """Per‑run context that replaces `config["configurable"]`.

    Additional attributes can be added later without touching node signatures.
    """
    language: str = "English"

In [17]:
from langgraph.runtime import Runtime

def call_model(state: OverallState, runtime: Runtime[Context]):
    language = runtime.context.language
    system_message_content = "Respond in {}".format(language)
    system_message = SystemMessage(content=system_message_content)
    messages = [system_message, HumanMessage(content=state["question"])]
    response = model.invoke(messages)
    return {"answer": response}

In [24]:
workflow_ctx = StateGraph(
    state_schema=OverallState,
    context_schema=Context,        # <‑‑ NEW in v0.6
    input_schema=InputState,
    output_schema=OutputState,
)

workflow_ctx.add_edge(START, "agent")
workflow_ctx.add_node("agent", call_model)
workflow_ctx.add_edge("agent", END)

graph_ctx = workflow_ctx.compile()

In [22]:
workflow = StateGraph(ChatMessages)

workflow.add_edge(START, "agent")
workflow.add_node("agent", call_model)
workflow.add_edge("agent", END)

graph = workflow.compile()

In [25]:
context=Context(language="Spanish")
graph_ctx.invoke({"question": "What's the highest mountain in the world?"}, context=context)

{'answer': AIMessage(content='El monte Everest es la montaña más alta del mundo, con una altitud de 8,848 metros sobre el nivel del mar. Se encuentra en la cordillera del Himalaya, en la frontera entre Nepal y China.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 23, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dz8Ksfuch4vcuXZUXPLDBqejM8H05', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--77883950-81b9-43a5-9a1e-3d6574615147-0', usage_metadata={'input_tokens': 23, 'output_tokens': 49, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

In [26]:
context=Context(language="German")
graph_ctx.invoke({"question": "What's the highest mountain in the world?"}, context=context)

{'answer': AIMessage(content='Das höchste Berg der Welt ist der Mount Everest in der Himalaya-Region an der Grenze zwischen Nepal und Tibet.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 23, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dz8L83aSrkiZs9Lhtdc7i2Gjh5uOI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--d764e45c-87b6-4241-a296-8ec6a31edf11-0', usage_metadata={'input_tokens': 23, 'output_tokens': 26, 'total_tokens': 49, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})}